# web item

In [ ]:
from __future__ import annotations
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *

import random
import math
import pandas as pd

from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import string
from typing import NamedTuple


In [ ]:
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.overlay import OverlaySpec, FlowOverlay, CreamOverlay, RiverOverlay, OceanWaveOverlay
from HexMagic.climate import  TerraDemo, TerrainFactory
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns

In [ ]:


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve:  dict[int, float] = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list[float] = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


In [ ]:
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
class GameParts:

    def __init__(self, radius: int = 20, useKoreanMap: bool = False):
        if useKoreanMap:
            self.terr: TerraDemo = TerraDemo().japan_korea_map()
        else:
            world = TerrainFactory.create_world(
                bounds=MapRect(MapCord(0, 0), MapSize(300, 300)),
                preset='temperate',
                name='Maiden Lane',
                radius=15,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.3,
                oceanic_sides=['N'],
                terrain_age='young',
                formation_type='ridge',
                elevation_scale=1.5,
                erosion_age=0.1,
                num_lakes=0,
                seed=23,
                debug=True
            )

            self.terr = world.terrain
        self.grid: HexGrid = self.terr.hexGrid
        self.referenceIndex: int = self.grid.middle
        self.grid.adjustRadius(radius)
        rivers = self.terr.carve_to_ocean(num_lakes=0)
        self.builder: SVGBuilder = self.grid.builder
        self.basins: DrainageBasins = world.basins
        for i in range(len(self.grid.hexes)):
            self.grid.hexes[i].label = str(i)

        fy: FoodYield = FoodYield(self.terr, self.basins)
        
        self.yields: np.ndarray = fy.compute()
        self.fy: FoodYield = fy


In [ ]:

@patch
def hp2i(self: GameParts, hexpos: HexPosition) -> int:
    return self.grid.hexposition_to_index(hexpos=hexpos, origin_index=self.referenceIndex)

@patch
def i2hp(self: GameParts, index: int) -> HexPosition:
   return self.grid.index_to_hexposition(index=index, origin_index=self.referenceIndex)
        

In [ ]:
@patch
def overlayContext(parts: GameParts, *, region=None, padding=1, radius=None,
                   corridors=None, extra_squads=None, extra_pieces=None):
    """Build a GameContext from a GameParts instance."""
    terrain = parts.terr
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)

    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)

    # Collect all squads and pieces, merging extras if provided
    squads = list(getattr(parts, 'squads', []))
    if extra_squads:
        squads.extend(extra_squads)

    pieces = []
    for sq in squads:
        pieces.extend(sq.alive if hasattr(sq, 'alive') else sq.pieces)
    if extra_pieces:
        pieces.extend(extra_pieces)

    return GameContext(
    terrain=terrain, grid=grid, builder=grid.builder,
    c2f=getattr(terrain, 'c2f', None),
    extras=dict(
        basins=parts.basins,
        coarse_basins=parts.basins,
        corridors=corridors or [],
        pieces=pieces,
        squads=squads,
    )
)

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts(radius=30)

Done at iter 1: 0 lakes

=== TERRAIN FACTORY ===
Preset: Temperate
Description: Four seasons, moderate rainfall. Pacific Northwest-like.
Latitude: 35° to 55°
Longitude: -5.0° to 5.0°
Grid: 20 x 20 hexes
Terrain age: young, Formation: ridge
Base temperature range: 8°C to 14°C
Wind: 15.0 m/s from 270.0°
Done at iter 0: 0 lakes


In [ ]:
ctx = myStuff.overlayContext()
TerrainDisplay(
    CreamOverlay(stylized=True),
    RiverOverlay(max_width=4),
    FoodOverlay(),
    OceanWaveOverlay(
    num_waves=5,
    spacing=8,
    amplitude_start=3.0,
    amplitude_decay=0.6,
    wavelength=40,
    stroke_color="#4a7fb5",
    opacity_start=0.5,
    opacity_decay=0.7,
    stroke_width=1.2),
    ctx=ctx,
    debug = not showDemo
)

<div><div><strong>Untitled</strong><p class="text-sm">1043.25 × 885.0  ‹svg›</p></div><h4>Definitions</h4><table><thead><tr><th>ID</th><th>Tag</th><th>Len</th></tr></thead><tbody><tr><td>food_yield_1</td><td>pattern</td><td>153</td></tr><tr><td>food_yield_2</td><td>pattern</td><td>153</td></tr><tr><td>food_yield_3</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_4</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_5</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_6</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_7</td><td>pattern</td><td>138</td></tr></tbody></table><h4>Styles</h4><table><thead><tr><th>Name</th><th>Fill</th></tr></thead><tbody><tr><td><code>Foothills</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#FFCA28" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Highlands</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#FF9800" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Hills</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#D4E157" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Lowland</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#66BB6A" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Mountains</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#F57C00" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Peaks</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#BF360C" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Plains</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#9CCC65" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Snow</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#ffffff" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Summits</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#E65100" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>coast</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#F8F4E8" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>highlands</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#E8DFD0" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>hills</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#EFE6D5" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>lowland</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#FDF5E6" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>mountain</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#DED4C4" stroke="none" stroke-width="1" rx="3"><

In [ ]:
ctx = myStuff.overlayContext()
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    

    ctx=ctx,
    debug = not showDemo
)

<div><div><strong>Untitled</strong><p class="text-sm">1013.25 × 855.0  ‹svg›</p></div><h4>Definitions</h4><table><thead><tr><th>ID</th><th>Tag</th><th>Len</th></tr></thead><tbody><tr><td>food_yield_1</td><td>pattern</td><td>153</td></tr><tr><td>food_yield_2</td><td>pattern</td><td>153</td></tr><tr><td>food_yield_3</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_4</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_5</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_6</td><td>pattern</td><td>138</td></tr><tr><td>food_yield_7</td><td>pattern</td><td>138</td></tr></tbody></table><h4>Styles</h4><table><thead><tr><th>Name</th><th>Fill</th></tr></thead><tbody><tr><td><code>Foothills</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#FFCA28" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Highlands</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#FF9800" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Hills</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#D4E157" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Lowland</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#66BB6A" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Mountains</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#F57C00" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Peaks</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#BF360C" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Plains</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#9CCC65" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Snow</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#ffffff" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>Summits</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#E65100" stroke="#333333" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>coast</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#F8F4E8" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>highlands</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#E8DFD0" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>hills</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#EFE6D5" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>lowland</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#FDF5E6" stroke="none" stroke-width="1" rx="3"></rect></svg></td></tr><tr><td><code>mountain</code></td><td><svg xmlns="http://www.w3.org/2000/svg" viewbox="0 0 30 20" height="20" width="30"><rect width="28" height="18" x="0" y="0" fill="#DED4C4" stroke="none" stroke-width="1" rx="3"><

## Graph

In [ ]:
@dataclass
class MapPlace:
    location: HexPosition = HexPosition.origin()
    facing: HexPosition = HexPosition.W
    level: int = 0
    harvest: int = 0
    parent: GameParts | None = None


can ypu rewrite MapPlace with annotations

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠

Sure! Here's `MapPlace` with full type annotations:

```python
@dataclass
class MapPlace:
    location: HexPosition = HexPosition.origin()
    facing: HexPosition = HexPosition.W
    level: int = 0
    harvest: int = 0
    parent: GameParts | None = None
```

The key changes: `level`, `harvest`, and `parent` are now properly annotated. I used `GameParts | None` for `parent` since it defaults to `None`, which signals it's optional. If `level` or `harvest` should be `float` instead of `int`, just swap those out.

In [ ]:
@patch
def location(self:GameParts,location, facing = HexPosition.W):
    index = self.hp2i(location)
    retPlace=MapPlace(location,facing)
    retPlace.level = self.terr.elevationLevel(index)
    retPlace.harvest  = int(self.fy.tiers[index])
    retPlace.parent = self
    return retPlace

In [ ]:
@patch
def location_by_index(self:GameParts,index, facing = HexPosition.W):
    location = self.i2hp(index)
    return self.location(location,facing)

In [ ]:
myStuff.grid.middle, myStuff.grid.index_to_hexposition(myStuff.grid.middle), myStuff.i2hp(myStuff.grid.middle)

(210, HexPosition(10, 0, -10), HexPosition(0, 0, 0))

In [ ]:
myStuff.grid.hp2i??

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.harvest

(2, 3)

In [ ]:
@patch
def sight(self:MapPlace) -> HexRegion:
    """All hex indices visible from this piece's location."""
  
    effective_sight = max(2,self.level)
      
    retRegion = HexRegion(
        hexes=set([self.parent.hp2i(hp) for hp in self.location.field_of_view( self.facing, effective_sight)]),
        hexGrid = self.parent.grid)

    return retRegion


In [ ]:
place.sight()

HexRegion(hexes={354, 355, 295, 334, 335, 375, 314, 315}, hexGrid=<HexMagic.plot.hex.HexGrid object at 0x71620f6d9910>)

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.sight()

(2,
 HexRegion(hexes={354, 355, 295, 334, 335, 375, 314, 315}, hexGrid=<HexMagic.plot.hex.HexGrid object at 0x71620f6d9910>))

In [ ]:
place = myStuff.location_by_index(273)
place.level, place.harvest

(4, 2)

## Pieces

In [ ]:
@dataclass
class Piece:
    id:str = ""
    flag:CountryFlag | None = None
    type:PieceType = PieceType.PAWN
    place:MapPlace | None = None
    name:str = ""
    year:int = 1900
    lantern = StyleCSS("blank",fill="#007",opacity=0.3)

In [ ]:
import httpx

@patch
def fetch_avatar(self: Piece, style: str = "avataaars") -> str:
    """Fetch and cache DiceBear SVG avatar using piece name as seed."""
    if not hasattr(self, '_avatar_svg') or self._avatar_svg is None:
        seed = self.name or f"piece_{id(self)}"
        resp = httpx.get(f"https://api.dicebear.com/9.x/{style}/svg?seed={seed}")
        resp.raise_for_status()
        self._avatar_svg = resp.text
    return self._avatar_svg

@patch
def avatar_id(self: Piece) -> str:
    return f"avatar_{self.name or id(self)}"

@patch
def register_avatar(self: Piece, builder: SVGBuilder, style: str = "avataaars"):
    """Register avatar as a <symbol> definition on the builder (idempotent)."""
    aid = self.avatar_id()
    # Skip if already registered
    if any(getattr(d, 'attributes', {}).get('id') == aid for d in builder.definitions):
        return
    svg_raw = self.fetch_avatar(style)
    # Extract viewBox from the SVG
    import re
    vb_match = re.search(r'viewBox="([^"]+)"', svg_raw)
    vb = vb_match.group(1) if vb_match else "0 0 280 280"
    # Strip outer <svg> tags to get inner content
    inner = re.sub(r'<svg[^>]*>', '', svg_raw)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    symbol = SVGDef("symbol", aid, inner, viewBox=vb)
    builder.add_definition(symbol)

@patch
def draw_avatar(self: Piece, center: MapCord, builder: SVGBuilder,
                size: float = 40, layer: str = None,
                opacity: float = 1.0, attrs: dict = None) -> str:
    """Render avatar at center with given size. Registers symbol if needed."""
    self.register_avatar(builder)
    aid = self.avatar_id()
    x, y = center.x - size / 2, center.y - size / 2
    extra = CountryFlag._render_attrs(attrs) if attrs else ""
    svg = (f'<use href="#{aid}" x="{x:.1f}" y="{y:.1f}" '
           f'width="{size:.1f}" height="{size:.1f}"{extra}/>')
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer:
        builder.adjust(layer, svg)
    return svg


In [ ]:
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int        # upkeep per turn while on board
    cost: int        # one-time cost to place
    harvest: float   # multiplier applied to tile's food yield tier
    drain: float     # cost imposed on EACH enemy piece in sight cone
    bonus_sight: int # extra sight range beyond elevation-based default

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=1, cost=5,  harvest=1.0, drain=0.5, bonus_sight=0),
    PieceType.BISHOP: FoodStats(diet=2, cost=12, harvest=0.5, drain=1.5, bonus_sight=2),
    PieceType.QUEEN:  FoodStats(diet=4, cost=25, harvest=0.0, drain=3.0, bonus_sight=1),
}

@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]


In [ ]:
@patch
def sight(self: Piece) -> HexRegion:
    """All hex indices visible from this piece's location."""
    bonus = self.food.bonus_sight
    effective_sight = max(2, self.place.level) + bonus
    
    return HexRegion(
        hexes=set([self.place.parent.hp2i(hp) 
                   for hp in self.place.location.field_of_view(self.place.facing, effective_sight)]),
        hexGrid=self.place.parent.grid)


what is place.level?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠

`place.level` is the **elevation level** of the hex at that location. It's set in `GameParts.location()`:

```python
retPlace.level = self.terr.elevationLevel(index)
```

So it comes from `Terrain.elevationLevel(index)`, which presumably buckets the raw elevation into a discrete tier (like 0=ocean, 1=lowland, 2=plains, 3=hills, 4=highlands, etc.).

It's used in two gameplay-relevant ways:

1. **Sight range** — in both `MapPlace.sight()` and `Piece.sight()`, the effective sight distance is `max(2, level)` (plus any `bonus_sight`). Higher elevation = see further.
2. **General terrain info** — paired with `harvest` to characterize what a hex offers strategically.

In your test, `location_by_index(336)` gave `level=2` and `location_by_index(273)` gave `level=4`, so hex 273 is at a notably higher elevation and would have a larger sight cone.

In [ ]:
@dataclass
class Squad:
    pieces: list[Piece] = field(default_factory=list)
    name: str = ""
    year: int = 1900
    animal: str = ""
    flag: CountryFlag | None = None
    countryName: str = ""
    storage:int = 100

    @classmethod
    def squads(cls, count: int = 4, flag: CountryFlag | None = None) -> list['Squad']:
        if flag is None:
            flag = CountryFlag.seaborn("husl", 1)[0]
        return [Squad(name=name, animal=animal, flag=flag) 
                for animal, name in flag.country_squads(count)]

In [ ]:
@patch
def make_squad(self: CountryFlag, piece_types: list[PieceType], year: int, seed: int = 0) -> Squad:
    """Create a Squad with animal name and age-appropriate named Pieces."""
    animal = self.animal_name()
    squad_nm = self.squad_name(animal, seed)
    
    rng = random.Random(hash((self.name, year, seed)))
    
    pieces = []
    for pt in piece_types:
        birth_year = year + rng.randint(20, 38)
        names = self.commonNames(year=birth_year)
        name = rng.choice(names)
        player = Piece(flag=self, type=pt, name=name, year=birth_year)
        player.id = ''.join(random.choices(string.ascii_letters, k=16))
        player.lantern.name = f"{player.id}_lantern"
        pieces.append(player)

    
    return Squad(pieces=pieces, name=squad_nm, year=year, animal=animal, flag=self)


In [ ]:
@patch
def loadSquads(self:GameParts, year: int=1880, seed: int = 0):
    self.flags = CountryFlag.seaborn("bright",levels=3)
    self.squads = []
    for i,flag in enumerate(self.flags):
        squad = flag.make_squad([PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.PAWN,PieceType.BISHOP,PieceType.BISHOP,PieceType.QUEEN], year=year, seed =seed) 
        self.squads.append(squad)

In [ ]:
myStuff.loadSquads()

In [ ]:
myStuff.squads[0].pieces[0]

Piece(id='TJcBSXaurGLvDKqp', flag=<HexMagic.game.flag.CountryFlag object at 0x71620ef43f20>, type=<PieceType.PAWN: 'pawn'>, place=None, name='Fred', year=1902)

In [ ]:
from fasthtml.common import *
from fasthtml.components import Uk_input_tag
from fasthtml.svg import *
from monsterui.all import *

In [ ]:
read_url("https://monsterui.answer.ai/cards/md")

'"""FrankenUI Cards Example built with MonsterUI (original design by ShadCN)"""\n\nfrom fasthtml.common import *\nfrom fasthtml.components import Uk_input_tag\nfrom fasthtml.svg import *\nfrom monsterui.all import *\nimport calendar\nfrom datetime import datetime\n\napp, rt = fast_app(hdrs=Theme.blue.headers())\n\nCreateAccount = Card(\n    Grid(Button(DivLAligned(UkIcon(\'github\'),Div(\'Github\'))),Button(\'Google\')),\n            DividerSplit("OR CONTINUE WITH", text_cls=TextPresets.muted_sm),\n            LabelInput(\'Email\',    id=\'email\',   placeholder=\'m@example.com\'),\n            LabelInput(\'Password\', id=\'password\',placeholder=\'Password\', type=\'Password\'),\n            header=(H3(\'Create an Account\'),Subtitle(\'Enter your email below to create your account\')),\n            footer=Button(\'Create Account\',cls=(ButtonT.primary,\'w-full\')))\n\nPaypalSVG_data = "M7.076 21.337H2.47a.641.641 0 0 1-.633-.74L4.944.901C5.026.382 5.474 0 5.998 0h7.46c2.57 0 4.578.543

In [ ]:

@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    return Card(
        DivCentered(
            NotStr(b.xml()),
            P(self.name, cls=(TextT.sm, TextT.medium)),
            P(f"{self.type.icon} · {self.year}", cls=TextPresets.muted_sm)),
        cls=CardT.hover)


In [ ]:
myStuff.loadSquads()
show(myStuff.squads[0].pieces[0])

HTML(<div class="uk-card uk-card hover:shadow-lg hover:-translate-y-1 transition-all duration-200">
  <div class="uk-card-body space-y-6">
    <div class="flex flex-col items-center justify-center space-y-4">
<?xml version='1.0' encoding='utf-8'?>
<svg  width="60" height="60" viewBox="0 0 60 60" xmlns="http://www.w3.org/2000/svg">
<title> Untitled </title>
  <defs>
    <symbol viewBox="0 0 280 280" id="avatar_William"><metadata xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/"><rdf:RDF><rdf:Description><dc:title>Avataaars</dc:title><dc:creator>Pablo Stanley</dc:creator><dc:source xsi:type="dcterms:URI">https://avataaars.com/</dc:source><dcterms:license xsi:type="dcterms:URI">https://avataaars.com/</dcterms:license><dc:rights>Remix of „Avataaars” (https://avataaars.com/) by „Pablo Stanley”, licensed under „Free for personal and commercial use” (https://avataaars.com/)</dc:rights></rdf:Description></rdf:RDF></metadata><mask id="viewboxMask"><rect width="280" height="280" rx="0" ry="0" x="0" y="0" fill="#fff" /></mask><g mask="url(#viewboxMask)"><g transform="translate(8)"><path d="M132 36a56 56 0 0 0-56 56v6.17A12 12 0 0 0 66 110v14a12 12 0 0 0 10.3 11.88 56.04 56.04 0 0 0 31.7 44.73v18.4h-4a72 72 0 0 0-72 72v9h200v-9a72 72 0 0 0-72-72h-4v-18.39a56.04 56.04 0 0 0 31.7-44.73A12 12 0 0 0 198 124v-14a12 12 0 0 0-10-11.83V92a56 56 0 0 0-56-56Z" fill="#ffdbb4"/><path d="M108 180.61v8a55.79 55.79 0 0 0 24 5.39c8.59 0 16.73-1.93 24-5.39v-8a55.79 55.79 0 0 1-24 5.39 55.79 55.79 0 0 1-24-5.39Z" fill="#000" fill-opacity=".1"/><g transform="translate(0 170)"><path d="M132.5 65.83c27.34 0 49.5-13.2 49.5-29.48 0-1.37-.16-2.7-.46-4.02A72.03 72.03 0 0 1 232 101.05V110H32v-8.95A72.03 72.03 0 0 1 83.53 32a18 18 0 0 0-.53 4.35c0 16.28 22.16 29.48 49.5 29.48Z" fill="#ff5c5c"/></g><g transform="translate(78 134)"><path fill-rule="evenodd" clip-rule="evenodd" d="M35.12 29.87a19 19 0 0 1 37.77.09c.08.77-.77 2.04-1.85 2.04H37.1C36 32 35 30.82 35.12 29.87Z" fill="#000" fill-opacity=".7"/><path d="M69.59 32H38.4a11 11 0 0 1 15.6-6.8A11 11 0 0 1 69.59 32Z" fill="#FF4F6D"/><path d="M66.57 17.75A5 5 0 0 1 65 18H44c-.8 0-1.57-.2-2.24-.53A18.92 18.92 0 0 1 54 13c4.82 0 9.22 1.8 12.57 4.75Z" fill="#fff"/></g><g transform="translate(104 122)"><path fill-rule="evenodd" clip-rule="evenodd" d="M16 8c0 4.42 5.37 8 12 8s12-3.58 12-8" fill="#000" fill-opacity=".16"/></g><g transform="translate(76 90)"><path d="M44 20.73c0 4.26-6.27 7.72-14 7.72S16 25 16 20.73C16 16.46 22.27 13 30 13s14 3.46 14 7.73ZM96 20.73c0 4.26-6.27 7.72-14 7.72S68 25 68 20.73C68 16.46 74.27 13 82 13s14 3.46 14 7.73Z" fill="#fff"/><path d="M32.82 28.3a25.15 25.15 0 0 1-5.64 0 6 6 0 1 1 5.64 0ZM84.82 28.3a25.15 25.15 0 0 1-5.64 0 6 6 0 1 1 5.64 0Z" fill="#000" fill-opacity=".7"/></g><g transform="translate(76 82)"><path d="M36.37 6.88c-1.97 2.9-5.55 4.64-8.74 5.68-3.94 1.29-18.55 3.38-15.11 11.35.05.12.22.12.27 0 1.15-2.65 17.47-5.12 18.97-5.7 4.45-1.71 8.4-5.5 9.17-10.55.35-2.31-.64-6.05-1.55-7.55-.11-.18-.37-.13-.43.07-.36 1.33-1.41 4.97-2.58 6.7ZM75.63 6.88c1.97 2.9 5.55 4.64 8.74 5.68 3.94 1.29 18.55 3.38 15.11 11.35a.15.15 0 0 1-.27 0c-1.15-2.65-17.47-5.12-18.97-5.7-4.45-1.71-8.4-5.5-9.17-10.55-.35-2.31.64-6.05 1.55-7.55.11-.18.37-.13.43.07.36 1.33 1.41 4.97 2.58 6.7Z" fill-rule="evenodd" clip-rule="evenodd" fill="#000" fill-opacity=".6"/></g><g transform="translate(-1)"><path fill-rule="evenodd" clip-rule="evenodd" d="M66 77.34c-.66 3.79-1 7.68-1 11.66v48c0 .97.02 1.94.06 2.9L65 142c.14 3.68-1.86 11.8-4.34 21.9-3.88 15.77-8.94 36.4-8.94 52.55 0 13.01 1.98 22.84 3.89 32.3 1.97 9.78 3.86 19.16 3.39 31.25h47s-.95-13.2-2.47-26.36c10.05 10.2 22.82 16.84 39.05 16.84 70.55 0 77.62-53.83 77.62-65.24 0-6.04-4.32-10.88-8.39-15.44-3.6-4.05-7.02-7.87-7-12.1 0-4.35 1.02-7.39 2.07-10.52 1.12-3.33 2.27-6.75 2.27-11.96 0-5.82-1.43-7.5-2.9-9.25a10.7 10.7 0 0 1-2.8

In [ ]:
show(myStuff.squads[0].pieces[0])

HTML(<div class="uk-card uk-card hover:shadow-lg hover:-translate-y-1 transition-all duration-200">
  <div class="uk-card-body space-y-6">
    <div class="flex flex-col items-center justify-center space-y-4">
<?xml version='1.0' encoding='utf-8'?>
<svg  width="60" height="60" viewBox="0 0 60 60" xmlns="http://www.w3.org/2000/svg">
<title> Untitled </title>
  <defs>
    <symbol viewBox="0 0 280 280" id="avatar_William"><metadata xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/"><rdf:RDF><rdf:Description><dc:title>Avataaars</dc:title><dc:creator>Pablo Stanley</dc:creator><dc:source xsi:type="dcterms:URI">https://avataaars.com/</dc:source><dcterms:license xsi:type="dcterms:URI">https://avataaars.com/</dcterms:license><dc:rights>Remix of „Avataaars” (https://avataaars.com/) by „Pablo Stanley”, licensed under „Free for personal and commercial use” (https://avataaars.com/)</dc:rights></rdf:Description></rdf:RDF></metadata><mask id="viewboxMask"><rect width="280" height="280" rx="0" ry="0" x="0" y="0" fill="#fff" /></mask><g mask="url(#viewboxMask)"><g transform="translate(8)"><path d="M132 36a56 56 0 0 0-56 56v6.17A12 12 0 0 0 66 110v14a12 12 0 0 0 10.3 11.88 56.04 56.04 0 0 0 31.7 44.73v18.4h-4a72 72 0 0 0-72 72v9h200v-9a72 72 0 0 0-72-72h-4v-18.39a56.04 56.04 0 0 0 31.7-44.73A12 12 0 0 0 198 124v-14a12 12 0 0 0-10-11.83V92a56 56 0 0 0-56-56Z" fill="#ffdbb4"/><path d="M108 180.61v8a55.79 55.79 0 0 0 24 5.39c8.59 0 16.73-1.93 24-5.39v-8a55.79 55.79 0 0 1-24 5.39 55.79 55.79 0 0 1-24-5.39Z" fill="#000" fill-opacity=".1"/><g transform="translate(0 170)"><path d="M132.5 65.83c27.34 0 49.5-13.2 49.5-29.48 0-1.37-.16-2.7-.46-4.02A72.03 72.03 0 0 1 232 101.05V110H32v-8.95A72.03 72.03 0 0 1 83.53 32a18 18 0 0 0-.53 4.35c0 16.28 22.16 29.48 49.5 29.48Z" fill="#ff5c5c"/></g><g transform="translate(78 134)"><path fill-rule="evenodd" clip-rule="evenodd" d="M35.12 29.87a19 19 0 0 1 37.77.09c.08.77-.77 2.04-1.85 2.04H37.1C36 32 35 30.82 35.12 29.87Z" fill="#000" fill-opacity=".7"/><path d="M69.59 32H38.4a11 11 0 0 1 15.6-6.8A11 11 0 0 1 69.59 32Z" fill="#FF4F6D"/><path d="M66.57 17.75A5 5 0 0 1 65 18H44c-.8 0-1.57-.2-2.24-.53A18.92 18.92 0 0 1 54 13c4.82 0 9.22 1.8 12.57 4.75Z" fill="#fff"/></g><g transform="translate(104 122)"><path fill-rule="evenodd" clip-rule="evenodd" d="M16 8c0 4.42 5.37 8 12 8s12-3.58 12-8" fill="#000" fill-opacity=".16"/></g><g transform="translate(76 90)"><path d="M44 20.73c0 4.26-6.27 7.72-14 7.72S16 25 16 20.73C16 16.46 22.27 13 30 13s14 3.46 14 7.73ZM96 20.73c0 4.26-6.27 7.72-14 7.72S68 25 68 20.73C68 16.46 74.27 13 82 13s14 3.46 14 7.73Z" fill="#fff"/><path d="M32.82 28.3a25.15 25.15 0 0 1-5.64 0 6 6 0 1 1 5.64 0ZM84.82 28.3a25.15 25.15 0 0 1-5.64 0 6 6 0 1 1 5.64 0Z" fill="#000" fill-opacity=".7"/></g><g transform="translate(76 82)"><path d="M36.37 6.88c-1.97 2.9-5.55 4.64-8.74 5.68-3.94 1.29-18.55 3.38-15.11 11.35.05.12.22.12.27 0 1.15-2.65 17.47-5.12 18.97-5.7 4.45-1.71 8.4-5.5 9.17-10.55.35-2.31-.64-6.05-1.55-7.55-.11-.18-.37-.13-.43.07-.36 1.33-1.41 4.97-2.58 6.7ZM75.63 6.88c1.97 2.9 5.55 4.64 8.74 5.68 3.94 1.29 18.55 3.38 15.11 11.35a.15.15 0 0 1-.27 0c-1.15-2.65-17.47-5.12-18.97-5.7-4.45-1.71-8.4-5.5-9.17-10.55-.35-2.31.64-6.05 1.55-7.55.11-.18.37-.13.43.07.36 1.33 1.41 4.97 2.58 6.7Z" fill-rule="evenodd" clip-rule="evenodd" fill="#000" fill-opacity=".6"/></g><g transform="translate(-1)"><path fill-rule="evenodd" clip-rule="evenodd" d="M66 77.34c-.66 3.79-1 7.68-1 11.66v48c0 .97.02 1.94.06 2.9L65 142c.14 3.68-1.86 11.8-4.34 21.9-3.88 15.77-8.94 36.4-8.94 52.55 0 13.01 1.98 22.84 3.89 32.3 1.97 9.78 3.86 19.16 3.39 31.25h47s-.95-13.2-2.47-26.36c10.05 10.2 22.82 16.84 39.05 16.84 70.55 0 77.62-53.83 77.62-65.24 0-6.04-4.32-10.88-8.39-15.44-3.6-4.05-7.02-7.87-7-12.1 0-4.35 1.02-7.39 2.07-10.52 1.12-3.33 2.27-6.75 2.27-11.96 0-5.82-1.43-7.5-2.9-9.25a10.7 10.7 0 0 1-2.8

In [ ]:
@patch
def __ft__(self: Squad):
    # Header: flag banner + squad name + animal
    flag_builder = self.flag.flag_banner(width=210, height=120, wavy=True)

    # Animal icon
    ab = SVGBuilder()
    ab.width, ab.height = 250,250
    ab.adjust("animal", self.flag.animal_svg(
        size='large', animal=self.animal,
        center=MapCord(125, 125),
        piece_id=f"sq_{self.name}"))


    header = DivFullySpaced(
       # NotStr(flag_builder.xml()),
        H3(self.name),
        NotStr(ab.xml()),
        

        cls=TextPresets.muted_sm)

    # Group pieces by type
    from collections import defaultdict
    by_type = defaultdict(list)
    for p in self.pieces:
        by_type[p.type].append(p)

    sections = []
    for pt, pieces in by_type.items():
        # Section header with piece icon
        sec_header = DivLAligned(
            Span(pt.icon, cls=TextT.lg),
            H4(pt.value.title()),
            P(f"× {len(pieces)}", cls=TextPresets.muted_sm))
        # 4-wide grid of piece cards
        grid = Grid(*[p.__ft__() for p in pieces],
                    cols_sm=2, cols_md=4)
        sections.append(Div(sec_header, grid, cls='space-y-2'))

    return DivCentered(header,Card( *sections, cls='space-y-4'))


In [ ]:
@patch
def table(self: Squad):
    rows = []
    for p in self.pieces:
        loc = ""
        facing = ""
        if p.place and p.place.parent:
            loc = str(p.place.parent.hp2i(p.place.location))
            facing = str(p.place.facing.label)
        rows.append(Tr(
            Td(p.type.icon),
            Td(p.name, cls=TextT.medium),
            Td(str(p.year), cls=TextPresets.muted_sm),
            Td(loc, cls=TextPresets.muted_sm),
            Td(facing, cls=TextPresets.muted_sm),
        ))
    
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Born"), Th("Loc"), Th("Facing"))),
            Tbody(*rows),
        ),
        header=(H4(self.name), P(f"🐾 {self.animal} · {len(self.pieces)} pieces", cls=TextPresets.muted_sm)),
    )


In [ ]:
myStuff.loadSquads()
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

,Name,Born,Loc,Facing
♟,Edward,1912,,
♟,John,1915,,
♟,Paul,1908,,
♟,Walter,1906,,
♟,Raymond,1911,,
♝,Albert,1909,,
♝,Paul,1907,,
♛,John,1914,,


In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
show(unit)

HTML(<div class="uk-card uk-card hover:shadow-lg hover:-translate-y-1 transition-all duration-200">
  <div class="uk-card-body space-y-6">
    <div class="flex flex-col items-center justify-center space-y-4">
<?xml version='1.0' encoding='utf-8'?>
<svg  width="60" height="60" viewBox="0 0 60 60" xmlns="http://www.w3.org/2000/svg">
<title> Untitled </title>
  <defs>
    <symbol viewBox="0 0 280 280" id="avatar_Edward"><metadata xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/"><rdf:RDF><rdf:Description><dc:title>Avataaars</dc:title><dc:creator>Pablo Stanley</dc:creator><dc:source xsi:type="dcterms:URI">https://avataaars.com/</dc:source><dcterms:license xsi:type="dcterms:URI">https://avataaars.com/</dcterms:license><dc:rights>Remix of „Avataaars” (https://avataaars.com/) by „Pablo Stanley”, licensed under „Free for personal and commercial use” (https://avataaars.com/)</dc:rights></rdf:Description></rdf:RDF></metadata><mask id="viewboxMask"><rect width="280" height="280" rx="0" ry="0" x="0" y="0" fill="#fff" /></mask><g mask="url(#viewboxMask)"><g transform="translate(8)"><path d="M132 36a56 56 0 0 0-56 56v6.17A12 12 0 0 0 66 110v14a12 12 0 0 0 10.3 11.88 56.04 56.04 0 0 0 31.7 44.73v18.4h-4a72 72 0 0 0-72 72v9h200v-9a72 72 0 0 0-72-72h-4v-18.39a56.04 56.04 0 0 0 31.7-44.73A12 12 0 0 0 198 124v-14a12 12 0 0 0-10-11.83V92a56 56 0 0 0-56-56Z" fill="#f8d25c"/><path d="M108 180.61v8a55.79 55.79 0 0 0 24 5.39c8.59 0 16.73-1.93 24-5.39v-8a55.79 55.79 0 0 1-24 5.39 55.79 55.79 0 0 1-24-5.39Z" fill="#000" fill-opacity=".1"/><g transform="translate(0 170)"><path d="M196 38.63V110H68V38.63a71.52 71.52 0 0 1 26-8.94v44.3h76V29.69a71.52 71.52 0 0 1 26 8.94Z" fill="#ffafb9"/><path d="M86 83a5 5 0 1 1-10 0 5 5 0 0 1 10 0ZM188 83a5 5 0 1 1-10 0 5 5 0 0 1 10 0Z" fill="#F4F4F4"/></g><g transform="translate(78 134)"><rect x="22" y="7" width="64" height="26" rx="13" fill="#000" fill-opacity=".6"/><rect x="24" y="9" width="60" height="22" rx="11" fill="#fff"/><path d="M24.18 18H32V9.41A11 11 0 0 1 35 9h1v9h9V9h4v9h9V9h4v9h9V9h2c.68 0 1.35.06 2 .18V18h8.82l.05.28v3.44l-.05.28H75v8.82c-.65.12-1.32.18-2 .18h-2v-9h-9v9h-4v-9h-9v9h-4v-9h-9v9h-1a11 11 0 0 1-3-.41V22h-7.82a11.06 11.06 0 0 1 0-4Z" fill="#E6E6E6"/></g><g transform="translate(104 122)"><path fill-rule="evenodd" clip-rule="evenodd" d="M16 8c0 4.42 5.37 8 12 8s12-3.58 12-8" fill="#000" fill-opacity=".16"/></g><g transform="translate(76 90)"><path d="M35.96 10c-2.55 0-5.08 1.98-6.46 3.82-1.39-1.84-3.9-3.82-6.46-3.82-5.49 0-9.04 3.33-9.04 7.64 0 5.73 4.41 9.13 9.04 12.74 1.66 1.23 4.78 4.4 5.17 5.1.38.68 2.1.7 2.58 0 .48-.73 3.51-3.87 5.17-5.1 4.63-3.6 9.04-7 9.04-12.74 0-4.3-3.55-7.64-9.04-7.64ZM88.96 10c-2.55 0-5.08 1.98-6.46 3.82-1.39-1.84-3.9-3.82-6.46-3.82-5.49 0-9.04 3.33-9.04 7.64 0 5.73 4.41 9.13 9.04 12.74 1.65 1.23 4.78 4.4 5.17 5.1.38.68 2.1.7 2.58 0 .48-.73 3.51-3.87 5.17-5.1 4.63-3.6 9.04-7 9.04-12.74 0-4.3-3.55-7.64-9.04-7.64Z" fill="#FF5353" fill-opacity=".8"/></g><g transform="translate(76 82)"><g fill-rule="evenodd" clip-rule="evenodd" fill="#DADADA"><path d="M57 12.82ZM96.12 7.6c1.46.56 9.19 6.43 7.86 9.16a.8.8 0 0 1-1.29.22 10.63 10.63 0 0 0-1.7-1.19c-5.1-2.84-11.3-1.93-16.73-.91-6.12 1.14-12.11 3.48-18.39 2.67-2.04-.26-6.08-1.22-7.63-2.96-.47-.53-.06-1.38.64-1.43 1.44-.11 2.86-.86 4.33-1.28 3.65-1.03 7.4-1.56 11.11-2.29 6.62-1.3 15.17-4.53 21.8-2Z"/><path d="M58.76 12.76c-1.17.04-2.8 3.56-.56 3.68 2.23.11 1.73-3.72.56-3.68ZM55 12.8c0-.01 0-.01 0 0ZM15.88 7.56c-1.46.56-9.19 6.43-7.86 9.16.24.5.89.6 1.29.22.55-.52 1.58-1.11 1.71-1.18 5.1-2.84 11.3-1.93 16.73-.91 6.12 1.14 12.11 3.48 18.39 2.67 2.04-.26 6.08-1.22 7.63-2.96.47-.53.06-1.38-.64-1.43-1.44-.11-2.86-.86-4.33-1.28-3.65-1.03-7.4-1.56-11.11-2.29-6.62-1.3-15.17-4.53-21.8-2Z"/><path d="M54.97 11.79c1.17.04 2.77 4.5.53 4.67-2.24.18-1.7-4.71-.53-4.67Z"/></g></g><g transform="transla

In [ ]:
unit.place.sight()

HexRegion(hexes={269, 270, 271, 272, 290, 291, 292, 293, 310, 311, 312, 191, 331, 332, 211, 212, 351, 230, 231, 232, 250, 251, 252, 253}, hexGrid=<HexMagic.plot.hex.HexGrid object at 0x71620f6d9910>)

In [ ]:
unit.sight()

HexRegion(hexes={269, 270, 271, 272, 290, 291, 292, 293, 310, 311, 312, 191, 331, 332, 211, 212, 351, 230, 231, 232, 250, 251, 252, 253}, hexGrid=<HexMagic.plot.hex.HexGrid object at 0x71620f6d9910>)

In [ ]:
def FogOverlay( fill="#ffffff", stroke="#cccccc", **kw) -> OverlaySpec:
    """White out hexes with no terrain data."""
    def render(ctx: OverlayContext) -> str:
        c2f = getattr(ctx, 'c2f', None)
        if not c2f: return ""
        mapped = {ni for indices in c2f.values() for ni in (indices if isinstance(indices, list) else [indices])}
        fog = StyleCSS("fog", fill=fill, stroke=stroke, stroke_width=1)
        ctx.builder.add_style(fog)
        for i in range(len(ctx.grid.hexes)):
            if i not in mapped:
                ctx.grid.hexes[i].style = fog
        return ""
    return OverlaySpec("fog", render, priority=6)


In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(),
    
    

    ctx=ctx,
    debug = not showDemo
)

Name,Fill
Foothills,
Highlands,
Hills,
Lowland,
Mountains,
Peaks,
Plains,
Snow,
Summits,
fog,


In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(fill="purple"),
    

    ctx=ctx,
    debug = not showDemo
)

Name,Fill
Foothills,
Highlands,
Hills,
Lowland,
Mountains,
Peaks,
Plains,
Snow,
Summits,
fog,


In [ ]:
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)


In [ ]:
def PieceOverlay(**kw) -> OverlaySpec:
    """Render all pieces using their flag's chess-piece SVG."""
    def render(ctx) -> str:
        if not ctx.pieces:
            return ""

        grid = ctx.grid
        N = len(grid.hexes)
        scale = grid.radius * 0.8 / 22.5
        parts = []

        for i, piece in enumerate(ctx.pieces):
            if piece.place is None or piece.flag is None:
                continue

            coarse_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(coarse_idx):
                if 0 <= fi < N:
                    center = grid.hexes[fi].center
                    pid = f"pc_{piece.id}_{i}"

                    svg_str, pat_def = piece.flag.piece_svg(
                        piece.type,
                        MapCord(center.x, center.y),
                        scale=scale,
                        size='board',
                        piece_id=pid,
                    )

                    if pat_def is not None:
                        ctx.builder.add_definition(pat_def)

                    parts.append(svg_str)

        return '\n'.join(parts)

    return OverlaySpec("pieces", render, requires={'pieces'}, priority=80)


In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
sight = unit.location.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(),
    LanternOverlay(),
    PieceOverlay(),
    pieces=[unit],

    ctx=ctx,
    debug = not showDemo
)

Name,Fill
Foothills,
Highlands,
Hills,
Lowland,
Mountains,
Peaks,
Plains,
Snow,
Summits,
eSndUOUHEyXpzPUY_lantern,


In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
unit.place.facing = HexPosition.SE
sight = unit.location.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(),
    LanternOverlay(),
    PieceOverlay(),
    pieces=[unit],
    ctx=ctx,
    debug = not showDemo
)

Name,Fill
Foothills,
Highlands,
Hills,
Lowland,
Mountains,
Peaks,
Plains,
Snow,
Summits,
eSndUOUHEyXpzPUY_lantern,


In [ ]:
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

,Name,Born,Loc,Facing
♟,Edward,1912,273,SE
♟,John,1915,,
♟,Paul,1908,,
♟,Walter,1906,,
♟,Raymond,1911,,
♝,Albert,1909,,
♝,Paul,1907,,
♛,John,1914,,


In [ ]:
class MoveError(Exception):
    """Raised when a Move fails validation."""
    pass

In [ ]:
class Rotation(NamedTuple):
    piece: Piece
    facing: HexPosition
class Placement(NamedTuple):
    piece: Piece
    location: MapPlace
    
@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None

    @classmethod
    def from_squad(cls, squad: Squad, rotations: list[Rotation] = None, 
                placement: Placement | None = None) -> 'Move':
        """Validate and construct a Move from a squad's pieces."""
        rotations = rotations or []
        errors = []
        
        # All rotated pieces must belong to squad and already be placed
        for r in rotations:
            if r.piece not in squad.pieces:
                errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
            elif r.piece.place is None:
                errors.append(f"{r.piece.name}: cannot rotate — not yet placed")
        
        # Placed piece must belong to squad and NOT already be placed
        if placement is not None:
            if placement.piece not in squad.pieces:
                errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
            elif placement.piece.place is not None:
                errors.append(f"{placement.piece.name}: cannot place — already on the board")
        
        # A piece can't appear in both rotations and placement
        if placement is not None:
            rotated_pieces = {r.piece for r in rotations}
            if placement.piece in rotated_pieces:
                errors.append(f"{placement.piece.name}: appears in both rotation and placement")
        
        # Placement must be within sight of an existing squad piece (unless first piece)
        if placement is not None and placement.piece in squad.pieces:
            placed_allies = [p for p in squad.pieces if p.place is not None and p is not placement.piece]
            if placed_allies:
                target_idx = placement.location.parent.hp2i(placement.location.location)
                visible = set()
                for ally in placed_allies:
                    visible |= ally.place.sight().hexes
                if target_idx not in visible:
                    errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")
        
        if errors:
            raise MoveError("; ".join(errors))
        
        return Move(rotations=rotations, placement=placement)


In [ ]:
@patch
def apply(self: Move, parts: GameParts):
    """Apply this move to the board."""
    for r in self.rotations:
        if r.piece.place is None:
            raise MoveError(f"{r.piece.name}: can't rotate — not placed")
        r.piece.place.facing = r.facing
    
    if self.placement is not None:
        self.placement.piece.place = self.placement.location

In [ ]:
@dataclass
class MoveLog:
    parts: GameParts
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(
        columns=['turn', 'squad', 'piece', 'action', 'detail']))
    
    def record(self, squad: Squad, move: Move):
        """Validate, apply, and log."""
        validated = Move.from_squad(squad, move.rotations, move.placement)
        validated.apply(self.parts)
        turn = len(self._df)
        
        rows = []
        for r in validated.rotations:
            rows.append(dict(turn=turn, squad=squad.name, piece=r.piece.name,
                             action='rotate', detail=r.facing.label))
        if validated.placement:
            p = validated.placement
            idx = self.parts.hp2i(p.location.location)
            rows.append(dict(turn=turn, squad=squad.name, piece=p.piece.name,
                             action='place', detail=str(idx)))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)
    
    @property
    def df(self) -> pd.DataFrame: return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def turn_count(self) -> int:
        return int(self._df.turn.max() + 1) if len(self._df) else 0


So I was wondering about another validation. you can only place an item in another items site (unless you have no objects on the board)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great rule — "you must place within an ally's sight, unless it's your first piece." Here's the updated `from_squad`:

```python
@classmethod
def from_squad(cls, squad: Squad, rotations: list[Rotation] = None, 
            placement: Placement | None = None) -> 'Move':
    """Validate and construct a Move from a squad's pieces."""
    rotations = rotations or []
    errors = []
    
    # All rotated pieces must belong to squad and already be placed
    for r in rotations:
        if r.piece not in squad.pieces:
            errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
        elif r.piece.place is None:
            errors.append(f"{r.piece.name}: cannot rotate — not yet placed")
    
    # Placed piece must belong to squad and NOT already be placed
    if placement is not None:
        if placement.piece not in squad.pieces:
            errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
        elif placement.piece.place is not None:
            errors.append(f"{placement.piece.name}: cannot place — already on the board")
    
    # A piece can't appear in both rotations and placement
    if placement is not None:
        rotated_pieces = {r.piece for r in rotations}
        if placement.piece in rotated_pieces:
            errors.append(f"{placement.piece.name}: appears in both rotation and placement")
    
    # Placement must be within sight of an existing squad piece (unless first piece)
    if placement is not None and placement.piece in squad.pieces:
        placed_allies = [p for p in squad.pieces if p.place is not None and p is not placement.piece]
        if placed_allies:
            target_idx = placement.location.parent.hp2i(placement.location.location)
            visible = set()
            for ally in placed_allies:
                visible |= ally.place.sight().hexes
            if target_idx not in visible:
                errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")
    
    if errors:
        raise MoveError("; ".join(errors))
    
    return Move(rotations=rotations, placement=placement)
```

The key addition is near the bottom — it gathers the sight of all already-placed squad pieces into a `visible` set, then checks whether the target hex is in it. If `placed_allies` is empty (first piece), the check is skipped entirely.

So I am going to add another mechanic to the game of food. each piece on the board has a 
1. diet - how much it costs to keep them on the board
2 a cost - how much it costs to place them on the board
3 a harvest - how much they harvest based upon the yield of a tile
4 a drain - how much an opposing player has to pay for being in the cone.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice mechanic — it creates real tension between fielding expensive units and sustaining your economy. Here's a natural way to model it:

```python
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int      # upkeep per turn while on board
    cost: int      # one-time cost to place
    harvest: float # multiplier applied to tile's food yield tier
    drain: float   # cost imposed on each enemy piece in this piece's sight cone

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=1, cost=5,  harvest=1.0, drain=0.5),
    PieceType.BISHOP: FoodStats(diet=2, cost=12, harvest=0.5, drain=1.5),
    PieceType.QUEEN:  FoodStats(diet=4, cost=25, harvest=0.0, drain=3.0),
}
```

The idea being:
- **Pawns** are cheap and good harvesters — your bread-and-butter economy pieces
- **Bishops** cost more, harvest less, but their sight cone drains enemies harder
- **Queens** are pure military — expensive to field, no harvest, devastating drain

Then you can add a helper on `Piece` to look up its stats:

```python
@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]
```

Does the drain apply to *all* enemy pieces inside the cone, or just per-hex? And does `harvest` multiply against the tile's food tier directly (e.g. tier 5 × 1.0 = 5 food), or is there a different formula you're thinking?

I think it would be all pieces in the cone for the drain. we might also have a bonus sight for some pieces. bishops might have bonus sight

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Makes sense — bishops as long-range scouts/pressure pieces. Here's the updated model:

```python
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int        # upkeep per turn while on board
    cost: int        # one-time cost to place
    harvest: float   # multiplier applied to tile's food yield tier
    drain: float     # cost imposed on EACH enemy piece in sight cone
    bonus_sight: int # extra sight range beyond elevation-based default

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=1, cost=5,  harvest=1.0, drain=0.5, bonus_sight=0),
    PieceType.BISHOP: FoodStats(diet=2, cost=12, harvest=0.5, drain=1.5, bonus_sight=2),
    PieceType.QUEEN:  FoodStats(diet=4, cost=25, harvest=0.0, drain=3.0, bonus_sight=1),
}

@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]
```

And then update `sight` to use it:

```python
@patch
def sight(self: MapPlace, piece: Piece = None) -> HexRegion:
    """All hex indices visible from this location. Piece adds bonus_sight if given."""
    bonus = piece.food.bonus_sight if piece else 0
    effective_sight = max(2, self.level) + bonus
    
    return HexRegion(
        hexes=set([self.parent.hp2i(hp) 
                   for hp in self.location.field_of_view(self.facing, effective_sight)]),
        hexGrid=self.parent.grid)
```

Now a bishop on a hill (level 3) sees 3 + 2 = 5 deep, while a pawn on the same hill sees just 3. You'd call `piece.place.sight(piece)` instead of `piece.place.sight()` to get the bonus.

Want to sketch out the per-turn food accounting next — total harvest, total diet, total drain from enemies?